# Week 24: Geodesy: ellipsoid vs geoid, EGM2008

**Track:** Space GIS Architect (Expert)  
**Full primer + quiz:** [https://launchdetect.com/academy/week/24/](https://launchdetect.com/academy/week/24/)  
**Track index:** [https://launchdetect.com/academy/space-gis-architect/](https://launchdetect.com/academy/space-gis-architect/)

---

_GPS gives you 'height above ellipsoid'. Maps want 'height above mean sea level' (i.e. geoid). The two differ by up to ~100 m. This week is the science of fixing that._


## Why this week matters

GPS gives you ellipsoidal height, but maps and humans want orthometric height (mean sea level). The two can differ by 100 m globally. For any precise positioning work, you must convert.


## Learning objectives

By the end of this lab you will be able to:

- Distinguish reference ellipsoid (geometric) from geoid (physical)
- Apply EGM2008 corrections to elevations
- Understand why GPS altitudes need geoid correction for orthometric height
- Use proj-egm for vertical datum transformation


## Setup — and why these dependencies

This lab uses: `leafmap[common] xarray rasterio pystac-client boto3`. Run the cell below once; Colab installs into the runtime.


In [ ]:
# Install required packages
!pip install -q leafmap[common] xarray rasterio pystac-client boto3


## The approach (and why)

Use pyproj's compound-CRS support to convert ellipsoidal height (EPSG:4979) to orthometric height with EGM2008 geoid (EPSG:4326+5773). Test on Mt. Whitney where the expected geoid offset is documented.


In [ ]:
# Week 24: ellipsoid -> geoid (orthometric) height via pyproj + EGM2008.
from pyproj import Transformer

# EPSG:4979 = WGS84 3D (lat/lon/ellipsoidal-height)
# EPSG:4326+5773 = WGS84 lat/lon + EGM2008 vertical datum (orthometric height)
to_ortho = Transformer.from_crs("EPSG:4979", "EPSG:4326+5773", always_xy=True)

# Mt. Whitney GPS ellipsoidal-height example
lon, lat, h_ellipsoid = -118.6, 36.5, 4421.0  # meters above WGS84 ellipsoid

lon_o, lat_o, h_orthometric = to_ortho.transform(lon, lat, h_ellipsoid)
geoid_offset = h_ellipsoid - h_orthometric

print(f"Ellipsoidal height (GPS):  {h_ellipsoid:.2f} m")
print(f"Orthometric height (MSL):  {h_orthometric:.2f} m")
print(f"Geoid offset (N):          {geoid_offset:.2f} m")

# TODO: take a CSV of GPS-measured Sierra Nevada points, apply the conversion,
# compare with USGS-published orthometric heights.


## What just happened — and why it works

The geoid is the equipotential surface that best matches mean sea level globally. EGM2008 is a spherical-harmonic expansion to degree 2,159 that represents the geoid with ~15 cm accuracy. pyproj uses the model internally when you specify the +5773 vertical CRS.


## Common gotchas

- Vertical CRS support requires pyproj 3.x. Old versions silently use ellipsoidal heights instead.
- Compound CRS notation is finicky: 'EPSG:4326+5773' is correct; 'EPSG:4326,5773' is not.
- EGM2008 is what most modern data uses. Older data may be on EGM96 or NAVD88 (US datum-specific).


## Self-check

Verify your solution against these checks before considering the lab complete:

- [ ] Output type matches the expected format (GeoJSON / PNG / table / etc.).
- [ ] No exceptions raised on a clean run.
- [ ] Result is visually plausible — open the map cell, scan the values, sanity-check magnitudes.
- [ ] Quiz on the [week page](https://launchdetect.com/academy/week/24/) — try answering before checking the key.

---

Found a bug or want to contribute an improvement? Open an issue or PR at [github.com/launchdetect/academy-labs](https://github.com/launchdetect/academy-labs).
